# 03 · 멀티모달 Endpoint 배포 & 이미지 추론 — 멀티모달 추출

**TL;DR** — 학습한 멀티모달 모델을 vLLM DLC로 배포하고, 실제 영수증 이미지를 보내 JSON 추출을 확인합니다.

**Why** — gemma-4 멀티모달은 vLLM(≥0.19)이 이미지 입력(OpenAI 호환 image_url)을 지원합니다. 텍스트 트랙과 달리 🔴 이미지/오디오 입력을 막지 않고 그대로 서빙합니다(재-export 안 함).

**기존 Pain Point** — 텍스트 트랙에서 쓰던 `--limit-mm-per-prompt image=0` 을 여기서 쓰면 이미지가 막힙니다 — 멀티모달 트랙에선 그 플래그를 쓰지 않습니다.

> 🔴 실제 실행 시 AWS 자격증명·GPU·엔드포인트 과금이 발생합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

In [ ]:
import os, sys
# 리포 루트를 path에 추가해 common/ 와 트랙 로컬 모듈을 import
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO)
sys.path.insert(0, os.getcwd())

In [ ]:
import importlib, boto3
from common import config, dlc, aws_utils; importlib.reload(config)
from sagemaker.core.helper.session_helper import Session
from sagemaker.serve import ModelBuilder
import time
sess = Session(boto3.Session(region_name=config.AWS_REGION))
%store -r md_mm_extraction
%store -r model_data
%store -r role
model_data = globals().get('md_mm_extraction') or globals().get('model_data')
if 'role' not in dir() or not role or ':role/' not in str(role):
    role = config.resolve_sagemaker_role(sess)

# 리전 가드: %store 값이 옛 리전을 가리키면 자동 교체.
model_data = aws_utils.ensure_model_data_in_region(
    locals().get('model_data'), config.AWS_REGION, job_prefix='gemma-mm-extraction-train')
md_mm_extraction = model_data
%store model_data
%store md_mm_extraction
print('model_data:', model_data)
print('role      :', role)

## vLLM / SGLang / DJL LMI 로 멀티모달 배포
🔴 이미지 입력을 허용해야 하므로 이미지 개수 제한을 1 이상으로 둡니다(텍스트 트랙은 0으로 막지만 여기는 반대). gemma-4 서빙엔 vLLM ≥ 0.19가 필요하고 이 kit의 기본 이미지가 이를 충족합니다.

엔진은 `.env`의 `SERVING_ENGINE`이 결정하고 이미지 URI도 거기서 해석합니다 — 텍스트 트랙(`03_deploy_endpoint`)과 같은 방식입니다.

| `SERVING_ENGINE` | 이미지 허용 옵션 | 비고 |
|---|---|---|
| `vllm` (기본) | `SM_VLLM_LIMIT_MM_PER_PROMPT` | 접두사를 떼고 `--limit-mm-per-prompt`로 전달 |
| `sglang` | (기본 허용) | SGLang은 멀티모달 입력을 기본 허용 |
| `lmi` | `OPTION_LIMIT_MM_PER_PROMPT` | LMI는 `OPTION_*`를 **vLLM EngineArguments로 pass-through** |

> LMI도 멀티모달을 지원합니다 — 공식 vLLM user guide에 Qwen3-VL 예시와 `OPTION_LIMIT_MM_PER_PROMPT="{\"image\": 4, \"video\": 0}"`가 명시돼 있고, "LMI supports all additional vLLM EngineArguments in Pass-Through mode"라고 문서화돼 있습니다.
⚠️ 단 LMI는 **번들 vLLM 버전에 종속**됩니다 — gemma-4에는 vLLM ≥ 0.19가 필요하니 최신 LMI 태그를 쓰세요.

### 🔴 24GB GPU에서 배포가 `Failed`로 끝날 때 — CUDA OOM (실측 2026-07-31)
멀티모달 아티팩트는 **vision tower를 포함**하므로 텍스트 트랙보다 무겁습니다(실측 가중치 15.18 GiB). 여기에 vLLM 기본값 두 개가 겹치면 L4 24GB에서 엔진 초기화가 실패합니다.

**증상** — endpoint가 `Failed`, 이유는 `did not pass the ping health check`뿐입니다. 실제 원인은 CloudWatch 로그 안에 있습니다:
```
Available KV cache memory: 4.69 GiB
torch.OutOfMemoryError: CUDA out of memory. Tried to allocate 256.00 MiB.
  GPU 0 has a total capacity of 21.96 GiB of which 147.12 MiB is free.
  ...in flashinfer_sample -> top_k_mask_logits -> torch.empty_like(logits)
```

**왜 하필 256 MiB인가** — 이게 결정적 단서입니다. 샘플러의 logits 버퍼 크기가
`max_num_seqs × vocab_size × 4B = 256 × 262,144 × 4B = 정확히 256 MiB`입니다. 즉 **모델이 커서가 아니라, 동시 시퀀스 기본값(256)이 실습 규모에 비해 과하게 잡혀서** 터진 것입니다.

예산을 보면 왜 아슬아슬한지 보입니다(한도 = 21.96 × 0.92 = 20.21 GiB):

| 항목 | 크기 |
|---|---|
| 가중치(vision 포함) | 15.18 GiB |
| KV 캐시 (vLLM이 자동 배정) | 4.69 GiB |
| **남은 여유** | **0.34 GiB** |
| 실제로 더 필요한 양 (활성 0.27 + 비torch 0.07 + CUDAGraph 0.78) | 1.12 GiB → **0.78 GiB 부족** |

vLLM 자신도 로그에서 `--kv-cache-memory=3.76 GiB`를 권고합니다 — **KV를 4.69로 과대 배정한 것**입니다.

**대응** (아래 셀의 기본값):
- `MAX_NUM_SEQS=32` — logits 버퍼가 256 MiB → 32 MiB로 줄어듭니다. 실습은 동시 요청이 1~2건이라 손실이 없습니다.
- `GPU_MEM_UTIL=0.90` — KV 과대 배정을 막아 여유를 남깁니다.

🔴 **GPU를 바꿀 필요는 없습니다** — 실측으로 L4와 같은 절대 예산(20.2 GiB)으로 제한한 L40S에서 이 설정으로 **로드 + 이미지 추론이 정상 동작**했습니다(KV 3.36 GiB, 여유 1.54 GiB). 다만 동시 처리량이 필요하거나 `MAX_LEN`을 크게 늘릴 때는 `ml.g6e.2xlarge`(L40S 45GB)가 여유롭습니다 — 그 경우 `MAX_NUM_SEQS`를 다시 올리세요.
> ⚠️ 이 값들은 **컨테이너의 vLLM 버전에 따라 민감도가 다릅니다**. 같은 예산에서 0.25.1은 KV를 3.36 GiB로, 0.26.0은 4.69 GiB로 잡았습니다(실측). 그래서 버전에 기대지 않고 명시적으로 낮춰 둡니다.

In [ ]:
import json
# 🔴 엔진/이미지는 env가 결정합니다(SERVING_ENGINE, *_IMAGE_URI). 텍스트 트랙과 동일한 해석 경로.
ENGINE = config.SERVING_ENGINE
for name, uri in dlc.serving_image_table(config.AWS_REGION).items():
    print(('→ ' if name == ENGINE else '   ') + f'{name:8s} {uri}')
print()
endpoint_name = f'gemma-mm-extraction-{ENGINE}-{int(time.time())}'
serve_image = dlc.resolve_serving_image(config.AWS_REGION, ENGINE)
assert serve_image, f'{ENGINE} 이미지 해석 실패 — .env의 *_IMAGE_URI 를 확인하세요.'
print(f'{ENGINE} DLC image:', serve_image)

# 엔진별 env 키는 dlc.serving_env()가 관리. mm_limit=이미지 허용, max_num_seqs/mem_util=OOM 방지.
serve_env = dlc.serving_env(
    ENGINE,
    max_model_len=2048,
    max_num_seqs=32,
    gpu_memory_utilization='0.90',
    mm_limit=json.dumps({'image': 1}),
    hf_token=config.get_serving_hf_token(),   # gated 모델일 때만 채워집니다
)
print('serve_env:', serve_env)
mb = ModelBuilder(image_uri=serve_image, s3_model_data_url=model_data,
                  env_vars=serve_env, role_arn=role, sagemaker_session=sess,
                  instance_type=config.INFER_INSTANCE_TYPE)
mb.build()
endpoint = mb.deploy(endpoint_name=endpoint_name, initial_instance_count=1,
                     instance_type=config.INFER_INSTANCE_TYPE, wait=False)
ep_mm_extraction = endpoint_name   # 트랙 전용 키(전역은 다른 트랙이 덮어씀)
%store endpoint_name
%store ep_mm_extraction
from IPython.display import display
print('deploying:', endpoint_name)
display(aws_utils.cw_links(config.AWS_REGION, endpoint_name=endpoint_name))

## InService 대기 → 이미지 추론
endpoint가 InService가 되면, 영수증 이미지를 base64 data URL로 실어 OpenAI 호환 chat 스키마로 호출합니다.

### ⏸️ 세션이 끊겼다면 — 이 셀부터 이어서 실행
커널을 재시작했어도 **endpoint는 서버에 살아 있습니다.** 위 배포 셀을 다시 돌릴 필요 없이 이 셀만 실행하면 호출에 필요한 것(경로·import·`endpoint_name`)이 복구됩니다.

In [ ]:
# ── 세션 재개도 겸하는 셀 (이것만 실행하면 아래 추론 셀이 바로 동작) ──
import os, sys, importlib
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
for p in (REPO, os.getcwd()):
    if p not in sys.path:
        sys.path.insert(0, p)
from common import config, aws_utils; importlib.reload(config)
from sagemaker.core.resources import Endpoint

%store -r ep_mm_extraction
%store -r endpoint_name
endpoint_name = globals().get('ep_mm_extraction') or globals().get('endpoint_name')
assert endpoint_name, (
    'endpoint_name 이 없습니다. 위에서 배포하거나 직접 지정하세요:\n'
    "    endpoint_name = 'gemma-mm-extraction-vllm-...'")

ep = Endpoint.get(endpoint_name); ep.refresh()
if ep.endpoint_status != 'InService':
    print('waiting for InService (', ep.endpoint_status, ')...'); ep.wait_for_status(target_status='InService')
print('InService:', endpoint_name)

In [ ]:
import base64, io, json, time
import importlib, track_data as td; importlib.reload(td)
from common import config, aws_utils
from common.display_utils import show_image_inference

MAX_TOKENS = 768   # 정답 JSON 최대 592토큰(실측 100건) — 512로는 잘림

# samples/ 에 넣어 둔 영수증 2장을 즉시 로드합니다(데이터셋 다운로드 없음 — 실측 0.0초).
#    전량이 필요하면 td.load_seed_examples(n) — 캐시가 없으면 ~40초 걸립니다.
samples = td.load_sample_receipts()
sample = samples[0]        # samples[1] 로 바꾸면 메뉴가 더 많은 영수증
img = sample['image']
print(f"{sample['name']} | 메뉴 {sample['menu_items']}개 | {img.size}")

# JPEG로 전송 — payload가 PNG의 1/8(추론 시간은 동일)
buf = io.BytesIO(); img.save(buf, format='JPEG', quality=85)
data_url = 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode()
messages = [{'role':'user','content':[
    {'type':'image_url','image_url':{'url': data_url}},
    {'type':'text','text': td.INSTRUCTION}]}]

t0 = time.time()
out = aws_utils.invoke_sagemaker_chat(endpoint_name, messages, region=config.AWS_REGION,
                                      max_tokens=MAX_TOKENS, temperature=0.1)
print(f'추론 {time.time() - t0:.1f}s')
assert out, '빈 응답 — CloudWatch endpoint 로그를 확인하세요.'
show_image_inference(img, out, title='영수증 → JSON 추출')

# samples/ 에는 정답도 함께 있어 눈으로 바로 대조할 수 있습니다.
print('정답(ground truth):', json.dumps(sample['ground_truth'], ensure_ascii=False))

✅ 이미지→JSON 추출을 확인했습니다. 정량 평가는 held-out 이미지로 JSON 필드 정확도를 재면 됩니다. 🔴 실습을 마치면 **99_cleanup.ipynb**로 endpoint를 삭제하세요(과금 중단).